In [ ]:
# ensemble : stack 
import os
from glob import glob
from tqdm import tqdm
import rasterio

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset, Subset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchmetrics.functional import structural_similarity_index_measure as ssim_index
from transformers import SegformerConfig, SegformerForSemanticSegmentation



import re
import numpy as np
import pandas as pd

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'

import warnings
from rasterio.errors import NotGeoreferencedWarning
warnings.simplefilter("ignore", NotGeoreferencedWarning)
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=r"Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated.*"
)


In [ ]:
# device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(42)
print(device)

In [ ]:
# directories
himachal_raster = 'datasets128/himachal_128/images'
himachal_mask = 'datasets128/himachal_128/masks' 

himlad_raster = 'datasets128/himachal_ladakh_128/images'
himlad_mask = 'datasets128/himachal_ladakh_128/masks'

sikkim_raster = 'datasets128/sikkim128/images'
sikkim_mask = 'datasets128/sikkim128/masks'

kashmir_raster = 'datasets128/kashmir128/images'
kashmir_mask = 'datasets128/kashmir128/masks'

uttrakhand_raster = 'datasets128/uttrakhand128/images'
uttrakhand_mask = 'datasets128/uttrakhand128/masks'

#loss function : focal + dice + tversky

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, y_pred, y_true):
        eps = 1e-7
        y_pred = torch.clamp(y_pred, eps, 1. - eps)  # prevent log(0)
        bce = - (y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred))
        pt = torch.where(y_true == 1, y_pred, 1 - y_pred)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce
        return focal_loss.mean()
        
class DiceLoss(nn.Module):
    def __init__(self):
        super(DiceLoss, self).__init__()

    def forward(self, y_pred, y_true):
        eps = 1e-7
        y_pred = y_pred.squeeze(1)
        y_true = y_true.squeeze(1)
        intersection = (y_pred * y_true).sum()
        union = y_pred.sum() + y_true.sum()
        dice = (2. * intersection + eps) / (union + eps)
        return 1 - dice
        
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3):
        super(TverskyLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, y_pred, y_true):
        eps = 1e-7
        y_pred = y_pred.squeeze(1)
        y_true = y_true.squeeze(1)
        TP = (y_pred * y_true).sum()
        FP = ((1 - y_true) * y_pred).sum()
        FN = (y_true * (1 - y_pred)).sum()
        tversky = (TP + eps) / (TP + self.alpha * FP + self.beta * FN + eps)
        return 1 - tversky
        
class CombinedLoss(nn.Module):
    def __init__(self, weights=(1.0, 0.5, 0.5)):
        super(CombinedLoss, self).__init__()
        self.focal = FocalLoss()
        self.dice = DiceLoss()
        self.tversky = TverskyLoss()
        self.w_focal, self.w_dice, self.w_tversky = weights

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)  # logits to probabilities
        loss = (
            self.w_focal * self.focal(probs, targets.float()) +
            self.w_dice * self.dice(probs, targets.float()) +
            self.w_tversky * self.tversky(probs, targets.float())
        )
        return loss

focal_dice_tversky = CombinedLoss(weights=(1.0, 0.5, 0.5))

In [ ]:

#-----------------------ResUNet-------------------------------
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        ) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return F.relu(out)

class ResUNet(nn.Module):
    def __init__(self, in_channels=18, out_channels=1):
        super().__init__()
        self.init_conv = nn.Conv2d(in_channels, 64, kernel_size=3, padding=1)
        self.enc1 = ResidualBlock(64, 64)
        self.enc2 = ResidualBlock(64, 128)
        self.enc3 = ResidualBlock(128, 256)
        self.enc4 = ResidualBlock(256, 512)
        self.bottleneck = ResidualBlock(512, 1024)
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec1 = ResidualBlock(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec2 = ResidualBlock(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ResidualBlock(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec4 = ResidualBlock(128, 64)
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.enc1(F.relu(self.init_conv(x)))
        x2 = self.enc2(F.max_pool2d(x1, 2))
        x3 = self.enc3(F.max_pool2d(x2, 2))
        x4 = self.enc4(F.max_pool2d(x3, 2))
        x5 = self.bottleneck(F.max_pool2d(x4, 2))
        x = self.up1(x5)
        x = self.dec1(torch.cat([x, x4], dim=1))
        x = self.up2(x)
        x = self.dec2(torch.cat([x, x3], dim=1))
        x = self.up3(x)
        x = self.dec3(torch.cat([x, x2], dim=1))
        x = self.up4(x)
        x = self.dec4(torch.cat([x, x1], dim=1))
        return self.final_conv(x)
    

In [ ]:
#-----------------------SegformerB4-------------------------------
config = SegformerConfig.from_pretrained("nvidia/segformer-b4-finetuned-ade-512-512")
config.num_channels = 18   # your 18-band input
config.num_labels = 1      # glacier vs background
config.image_size = 128 
segformerb4 = SegformerForSemanticSegmentation(config)

# from torchinfo import summary
# summary(segformerb4, input_size=(1,18,128,128))

In [ ]:
# importing model weights

resunet = ResUNet(in_channels = 18, out_channels = 1)
resunet.to(device)
resunet.load_state_dict(torch.load("Github weights/ResUNet(focalDiceTversky).pt"))
resunet.eval()

segformerb4.to(device)
segformerb4.load_state_dict(torch.load("Github weights/SegformerB4(focalDiceTversky).pt"))
segformerb4.eval()

In [ ]:
# Data loader
def extract_row_col(filename):
    """
    Extracts row and column indices from filenames like 'patch_r12_c5.tif'
    """
    match = re.search(r"r(\d+)_c(\d+)", os.path.basename(filename))
    if match:
        row = int(match.group(1))
        col = int(match.group(2))
        return (row, col)
    return (0, 0) 
    
class GlacierDataset(Dataset):
    def __init__(self, image_paths, mask_paths, top_bands=None, index_bands = None, transform=None):
        self.image_paths = sorted(glob(os.path.join(image_paths, "*.tif")), key=extract_row_col)
        self.mask_paths = sorted(glob(os.path.join(mask_paths, "*.tif")), key=extract_row_col)
        self.top_bands = top_bands if top_bands else list(range(1, 19))  # 1-based indexing
        # self.index_bands = index_bands if index_bands else []  # subset of top_bands
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.image_paths[idx]) as src:
            image = src.read(self.top_bands)  # shape: (B, H, W)
        with rasterio.open(self.mask_paths[idx]) as src:
            mask = src.read(1)  # shape: (H, W)
            # print(mask.shape)

        
        image = np.transpose(image, (1, 2, 0))  # (H, W, C)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]
        else:
            image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)
            mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return image, mask



ds_himachal = GlacierDataset(himachal_raster, himachal_mask, top_bands = None)
ds_himlad = GlacierDataset(himlad_raster, himlad_mask, top_bands = None)
ds_sikkim = GlacierDataset(sikkim_raster, sikkim_mask, top_bands = None)
ds_kashmir = GlacierDataset(kashmir_raster, kashmir_mask, top_bands = None)
ds_uttrakhand = GlacierDataset(uttrakhand_raster, uttrakhand_mask, top_bands = None)



def sample_subset(dataset, fraction):
    n = int(len(dataset) * fraction)
    indices = np.random.choice(len(dataset), n, replace=False).tolist() 
    return Subset(dataset, indices)

ds_himlad_sub = sample_subset(ds_himlad, 0.3)
ds_sikkim_sub = sample_subset(ds_sikkim, 0.3)
ds_kashmir_sub = sample_subset(ds_kashmir, 0.3)
ds_uttrakhand_sub = sample_subset(ds_uttrakhand, 0.3)

# Concatenate all datasets
combined_dataset = ConcatDataset([ds_himachal, ds_himlad_sub, ds_sikkim_sub, ds_kashmir_sub, ds_uttrakhand_sub])

val_ratio = 0.3
val_size = int(len(combined_dataset) * val_ratio)
train_size = len(combined_dataset) - val_size
train_dataset, val_dataset = random_split(combined_dataset, [train_size, val_size])

# DataLoaders
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=4)




In [ ]:
# Put models in eval mode to check the outputs so that there is no differences while stacking 
resunet.eval()
segformerb4.eval()

# Get a batch of data
images, masks = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    out1 = resunet(images)              # ResUNet raw output
    out2 = segformerb4(images).logits   # SegFormer raw output

print("ResUNet output:")
print("  shape:", out1.shape)
print("  min:", out1.min().item(), "max:", out1.max().item())
print("  mean:", out1.mean().item())

print("SegFormer output:")
print("  shape:", out2.shape)
print("  min:", out2.min().item(), "max:", out2.max().item())
print("  mean:", out2.mean().item())
# found  that both output are logits
# the shape of resunet output is (8, 1, 128, 128) while the shape of segformer output is (8, 1, 32, 32), segformer upsampled to 128x128

In [ ]:
# # stacking the logits 
# def stacked_output(x):
#     x = x.to(device)
#     with torch.no_grad():
#         out1 = resunet(x)               # logits [B,1,128,128]
#         out2 = segformerb4(x).logits    # logits [B,1,32,32]
#         # Upsample segformer logits to match ResUNet resolution
#         out2 = F.interpolate(out2, size=out1.shape[2:], mode="bilinear", align_corners=False)
#         stacked = torch.cat((out1, out2), dim=1)  # [B, 2, 128, 128]
#     return stacked

In [ ]:
## stacked output with augmentations such that model does not read noise much 
def stacked_output(image):
    # image: [1,C,H,W] on device
    preds = []
    transforms = [lambda x: x,
                  lambda x: torch.flip(x, [-1]),       # horiz
                  lambda x: torch.flip(x, [-2]),       # vert
                  lambda x: torch.rot90(x, 1, (-2,-1))]
    for fn in transforms:
        x = fn(image)
        out1 = resunet(x); out2 = segformerb4(x).logits
        out2 = F.interpolate(out2, size=out1.shape[2:], mode='bilinear')
        # revert transform on outputs
        out1 = fn(out1)   # flipping twice restores shape
        out2 = fn(out2)
        preds.append(torch.cat([out1, out2], dim=1))
    stacked = torch.mean(torch.stack(preds, 0), dim=0)
    return stacked


In [ ]:
# meta model to a tiny UNet 

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class MetaUNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=1):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(64, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        self.out_conv = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))

        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out_conv(d1)
meta_unet = MetaUNet(in_channels=2, out_channels=1).to(device)
 


In [ ]:
from sklearn.metrics import confusion_matrix

def train_metaUnet(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=30):
    best_val_loss = float("inf") 
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training"):
            images, masks = images.to(device), masks.to(device)
            stacked_preds = stacked_output(images) 
            optimizer.zero_grad()
            outputs = model(stacked_preds)   # logits
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        cm = np.zeros((2,2), dtype=int)
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Validation"):
                images, masks = images.to(device), masks.to(device)
                stacked_preds = stacked_output(images)
                outputs = model(stacked_preds)  # logits

                # loss
                loss = criterion(outputs, masks)
                val_loss += loss.item() * images.size(0)

                # prepare for confusion matrix
                preds = torch.sigmoid(outputs)
                preds = (preds > 0.5).long().cpu().numpy().flatten()
                true = masks.long().cpu().numpy().flatten()

                batch_cm = confusion_matrix(true, preds, labels=[0,1])
                cm += batch_cm

        val_loss /= len(val_loader.dataset)

        # IoU from confusion matrix
        tn, fp, fn, tp = cm.ravel()
        eps = 1e-7
        val_iou = (tp + eps) / (tp + fp + fn + eps)

        # scheduler step
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_loss)
        else:
            scheduler.step()

        print(f"Epoch {epoch+1}/{num_epochs}, "
              f"Train Loss: {train_loss:.4f}, "
              f"Val Loss: {val_loss:.4f}, "
              f"Val IoU: {val_iou:.4f}")

        # save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_meta_unet.pth")
            print(" Saved Best Model")


In [ ]:
# implementation training
optimizer = optim.Adam(meta_unet.parameters(), lr=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

train_metaUnet(meta_unet, train_loader, val_loader, focal_dice_tversky,optimizer, scheduler, num_epochs=30)

In [ ]:
meta_unet.load_state_dict(torch.load("Ensemble(30eps)(FocalDiceTversky).pth"))
meta_unet.eval()

In [ ]:
def evaluate(model, loader, return_predictions = False, plot_confusion_matrix = False):
    model.eval()

    if isinstance(model,nn.DataParallel):
        model = model.module
    criterion = focal_dice_tversky
    all_preds, all_gts, all_rasters = [], [], []

    metrics = {'loss':0.0, 'accuracy': 0, 'precision': 0, 'iou': 0,
                'recall':0, 'f1': 0, 'dice': 0, 'kappa':0}

    cm = np.zeros((2,2), dtype = int)
    with torch.no_grad():
        for X, Y in loader:
            X, Y = X.to(device), Y.to(device)
            stacked_preds = stacked_output(X) 
            outputs = model(stacked_preds)  # logits
            preds = (torch.sigmoid(outputs)>0.5).float()

            if return_predictions:
                all_preds.extend(preds.cpu())
                all_gts.extend(Y.cpu())
                all_rasters.extend(X.cpu())

            preds_np = preds.cpu().numpy().astype(bool).reshape(-1)
            Y_np = Y.cpu().numpy().astype(bool).reshape(-1)
            
            batch_cm = confusion_matrix(Y_np, preds_np, labels=[0, 1])
            cm += batch_cm

            metrics['loss'] += criterion(outputs, Y).item()

    eps = 1e-7
    tn, fp, fn, tp = cm.ravel()

    metrics['loss'] /= len(loader)
    metrics['iou'] = (tp + eps) / (tp + fp + fn + eps)
    metrics['accuracy'] = (tp + tn + eps) / (tp + tn + fp + fn + eps)
    metrics['precision'] = (tp + eps) / (tp + fp + eps)
    metrics['recall'] = (tp + eps) / (tp + fn + eps)
    metrics['f1'] = (2 * metrics['precision'] * metrics['recall']) / (metrics['precision'] + metrics['recall'] + eps)
    metrics['dice'] = (2 * tp + eps) / (2 * tp + fp + fn + eps)

    total = tp + tn + fp + fn
    po = (tp + tn) / total
    pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / (total ** 2)
    metrics['kappa'] = (po - pe) / (1 - pe + eps) if total > 0 else 0

    if plot_confusion_matrix:
        plt.figure(figsize=(5, 5))
        sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', 
                    xticklabels=['Background', 'Foreground'],
                    yticklabels=['Background', 'Foreground'])
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title('Confusion Matrix')
        plt.show()

    if return_predictions:
        return metrics, all_preds, all_gts, all_rasters, cm
    else:
        return metrics, cm



In [ ]:
def get_ordered_loader(image_dir, mask_dir, batch_size=4):
    
    dataset = GlacierDataset(image_dir, mask_dir,top_bands= None,transform=None)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=4)

test_loader1 = get_ordered_loader(himachal_raster, himachal_mask, batch_size=4)
test_loader2 = get_ordered_loader(himlad_raster, himlad_mask, batch_size=4)
test_loader3 = get_ordered_loader(sikkim_raster, sikkim_mask, batch_size=4)
test_loader4 = get_ordered_loader(kashmir_raster, kashmir_mask, batch_size=4)   
test_loader5 = get_ordered_loader(uttrakhand_raster, uttrakhand_mask, batch_size=4)

In [ ]:
test_metrics1, pred1, gt1, rt1, cm1 = evaluate(meta_unet, test_loader1, return_predictions = True)
test_metrics2, pred2, gt2, rt2, cm2 = evaluate(meta_unet, test_loader2, return_predictions = True)
test_metrics3, pred3, gt3, rt3, cm3 = evaluate(meta_unet, test_loader3, return_predictions = True)
test_metrics4, pred4, gt4, rt4, cm4 = evaluate(meta_unet, test_loader4, return_predictions = True)
test_metrics5, pred5, gt5, rt5, cm5 = evaluate(meta_unet, test_loader5, return_predictions = True)  

In [ ]:

Region_names = ['Himachal'.'Himachal Ladakh', 'Sikkim', 'Kashmir', 'Uttrakhand'] 

metrics_list = [test_metrics1, test_metrics2, test_metrics3, test_metrics4, test_metrics5] 

df = pd.DataFrame(metrics_list)

df.index = pd.Index(Region_names)
print(df)